# Home Health Visit-Note Feature Extraction — Simple Edition

A single-notebook version of the `hh-note-features` pipeline. Reads a few
synthetic Home Health visit notes, sends them to a **Gemini** model with a
small Pydantic schema, and returns typed, evidence-quoted features.

**What this notebook does:**
1. Sets up a Gemini client (Vertex AI for PHI-safe BAA mode, or direct API for dev).
2. Defines a compact `VisitNoteFeatures` schema (vitals, symptoms, risk flags, escalation).
3. Sends three sample visit notes through Gemini with a strict extraction prompt.
4. Shows the extracted features as a tidy DataFrame.

> ⚠️ The included sample notes are **synthetic**. Never run this on real PHI
> outside the Vertex AI BAA path.


## 1. Install dependencies

In [ ]:
# Run once. Comment out after first execution.
%pip install -q google-genai pydantic pandas python-dotenv

## 2. Configure the Gemini client

Two modes — pick one by setting environment variables before running this cell.

| Mode | Env vars | When to use |
|---|---|---|
| **Vertex AI** (BAA-covered, PHI-safe) | `USE_VERTEX_AI=true`, `GOOGLE_CLOUD_PROJECT=...` + ADC | Production, any real clinical data |
| **Direct Gemini API** (dev only) | `GEMINI_API_KEY=...` | Dev/testing with synthetic notes |

For Vertex, run `gcloud auth application-default login` in a terminal first.


In [ ]:
import os
from dotenv import load_dotenv
from google import genai

load_dotenv()  # picks up a local .env if you have one

USE_VERTEX = os.getenv("USE_VERTEX_AI", "").lower() in {"1", "true", "yes"}

if USE_VERTEX:
    project = os.environ["GOOGLE_CLOUD_PROJECT"]
    location = os.getenv("GOOGLE_CLOUD_LOCATION", "us-central1")
    client = genai.Client(vertexai=True, project=project, location=location)
    print(f"Using Vertex AI · project={project} · location={location}")
else:
    api_key = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
    if not api_key:
        raise RuntimeError(
            "Set GEMINI_API_KEY (dev) or USE_VERTEX_AI=true + GOOGLE_CLOUD_PROJECT (prod)."
        )
    client = genai.Client(api_key=api_key)
    print("Using direct Gemini API (dev mode — synthetic data only)")

MODEL = "gemini-2.5-flash"  # bump to gemini-2.5-pro for harder notes

## 3. Define the feature schema

A compact Pydantic model that captures the high-leverage signals from a Home Health
visit note: vitals, cardiopulmonary findings, mental status, falls, meds, and an
overall clinical impression. Every clinical finding carries a verbatim **evidence
quote** so the output is auditable.

The schema is intentionally smaller than the full feature-store version — easy to
extend, easy to reason about.


In [ ]:
from enum import Enum
from typing import Optional
from pydantic import BaseModel, Field


class Dyspnea(str, Enum):
    NONE = "none"
    ON_EXERTION = "on_exertion"
    AT_REST = "at_rest"
    NOT_DOCUMENTED = "not_documented"


class Status(str, Enum):
    STABLE = "stable"
    IMPROVING = "improving"
    DECLINING = "declining"
    ACUTE_CONCERN = "acute_concern"


class DxCluster(str, Enum):
    HF = "HF"
    COPD = "COPD"
    DM = "DM"
    POST_SURGICAL = "post_surgical"
    WOUND = "wound"
    OTHER = "other"


class Vitals(BaseModel):
    bp_systolic: Optional[int] = None
    bp_diastolic: Optional[int] = None
    heart_rate: Optional[int] = None
    spo2_percent: Optional[int] = None
    weight_lbs: Optional[float] = None
    weight_change_lbs: Optional[float] = Field(
        None, description="Positive=gain, negative=loss. Populate only if note states a delta."
    )


class Signal(BaseModel):
    name: str
    value: str
    evidence_quote: str = Field(description="Verbatim span from the note supporting this finding.")


class VisitNoteFeatures(BaseModel):
    primary_dx_cluster: DxCluster
    overall_status: Status
    vitals: Vitals
    dyspnea: Dyspnea
    dyspnea_evidence: Optional[str] = None
    new_confusion: bool = False
    confusion_evidence: Optional[str] = None
    fall_since_last_visit: bool = False
    prn_diuretic_used: bool = False
    adherence_concern: bool = False
    caregiver_present: bool = False
    escalation_indicators: list[str] = Field(
        default_factory=list,
        description="Findings warranting case-manager follow-up today.",
    )
    flat_signals: list[Signal] = Field(
        default_factory=list,
        description="5-10 most important findings with verbatim evidence quotes.",
    )
    rationale: str = Field(description="One-to-two-sentence justification of overall_status.")


# Quick sanity check
print("Schema fields:", list(VisitNoteFeatures.model_json_schema()["properties"].keys()))

## 4. Sample visit notes (synthetic)

Three notes covering different clinical scenarios:

1. **HFrEF early decompensation** — weight gain, JVD, new confusion. Should trigger escalation.
2. **Stable COPD** — routine follow-up, nothing acute. Should be `stable`.
3. **Uncontrolled diabetes + SDOH** — insulin gap, food insecurity. Adherence concern + escalation.


In [ ]:
NOTES = {
    "note_01_hfref": """
Patient: R.T. (synthetic). Visit type: Routine SN follow-up, episode day 14.
Primary dx: HFrEF (LVEF 28%), T2DM, CKD stage 3a.

Subjective: Pt reports increased SOB with ambulation to bathroom over past 3 days.
Denies chest pain. Daughter (primary caregiver, present at visit) states pt
"seems more confused than usual" for the past 2 days. Pt acknowledges taking
PRN furosemide 40 mg last night for "puffiness."

Objective: BP 158/92 (last visit 142/86). HR 96, RR 22, SpO2 93% on RA.
Weight 187 lbs — up 5.3 lbs from 7 days ago. Lungs: faint crackles bilateral
bases. JVD 1+ at 30 degrees. 2+ pitting edema bilateral lower extremities.

Plan: Notify HF case manager and PCP today. Increase visit frequency to 3x/week.
""",

    "note_02_copd_stable": """
Patient: J.W. (synthetic). Visit type: Routine SN follow-up, week 4.
Primary dx: COPD GOLD III. Lives with adult son (caregiver at work today).

Subjective: Pt stable per self-report. Reports continued use of albuterol rescue
inhaler "a couple times a day," consistent with baseline. Denies new sputum,
fever, or worsening cough.

Objective: BP 138/82, HR 88, RR 18, SpO2 92% on RA. Weight stable at 162 lbs.
Lungs: diminished breath sounds bilaterally with mild prolonged expiration.
No wheezes. Inhaler technique adequate after a quick reminder.

Plan: Continue current regimen. Next visit in 4 days.
""",

    "note_03_dm_sdoh": """
Patient: E.B. (synthetic). Visit type: Routine SN follow-up, week 6.
Primary dx: T2DM with peripheral neuropathy, A1c 9.4%. Lives alone; recently
widowed. No caregiver at visit.

Subjective: Glucose readings 78 to 312 mg/dL last week per log. Admits to
skipping evening insulin twice last week — "I ran out and didn't want to
bother my daughter for a ride." Endorses low mood, poor appetite, "some
nights I just don't eat — the fridge is mostly empty." Denies SI.

Objective: BP 152/88, HR 82, SpO2 98% on RA. Weight 218 lbs — down 4 lbs
(unintentional). Fingerstick at visit: 248 mg/dL. Insulin pen empty since
last Friday. PHQ-2 positive (score 5).

Plan: Notify PCP today — urgent insulin refill. Refer to social worker for
food insecurity and transportation. Increase visit frequency to 2x/week.
""",
}

for name, text in NOTES.items():
    print(f"{name}: {len(text)} chars")

## 5. The extraction prompt

A short system prompt that locks Gemini into conservative, evidence-cited
extraction. Two rules carry most of the weight:

- **Extract only what is documented.** Never infer findings from absence of mention.
- **Evidence quotes must be verbatim.** No paraphrasing.

The Pydantic schema is passed to Gemini via `response_schema`, so the model is
constrained to emit valid JSON matching our types.


In [ ]:
SYSTEM_PROMPT = """You are a clinical NLP extractor for a Home Health agency.
Read a single visit note and emit JSON matching the supplied schema.

RULES (non-negotiable):
1. Extract only what the note literally says. Never infer beyond the text.
   If a finding is not mentioned, leave the field null/false/not_documented.
2. Evidence quotes must be VERBATIM spans from the note. No paraphrasing.
3. Prefer structured values (e.g. "BP 158/92" -> bp_systolic=158, bp_diastolic=92).
4. escalation_indicators: only findings actually in the note that would prompt
   a case manager to act today. Do not pad this list.
5. flat_signals: the 5-10 most clinically important findings, each with a
   verbatim evidence_quote.
6. When unsure, under-extract. A missed signal is recoverable; a hallucinated
   signal corrupts the feature store.
"""

print(f"System prompt: {len(SYSTEM_PROMPT)} chars")

## 6. The extraction function

One function: takes note text, returns a validated `VisitNoteFeatures`. Uses
Gemini's structured-output mode with `response_schema` so we get typed objects
back, not strings to parse.


In [ ]:
from google.genai import types


def extract_features(note_text: str, model: str = MODEL) -> VisitNoteFeatures:
    """Send one visit note to Gemini and return a validated VisitNoteFeatures."""
    response = client.models.generate_content(
        model=model,
        contents=f'Visit note (verbatim):\n"""\n{note_text.strip()}\n"""\n\nExtract per the schema.',
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
            response_mime_type="application/json",
            response_schema=VisitNoteFeatures,
            temperature=0.0,  # extraction is not a creative task
        ),
    )
    # google-genai populates response.parsed when response_schema is a Pydantic class
    if isinstance(response.parsed, VisitNoteFeatures):
        return response.parsed
    return VisitNoteFeatures.model_validate_json(response.text)


# Smoke test on the first note
features = extract_features(NOTES["note_01_hfref"])
print("Primary dx :", features.primary_dx_cluster.value)
print("Status     :", features.overall_status.value)
print("Dyspnea    :", features.dyspnea.value, f"   (evidence: {features.dyspnea_evidence!r})")
print("BP         :", f"{features.vitals.bp_systolic}/{features.vitals.bp_diastolic}")
print("Wt delta   :", features.vitals.weight_change_lbs, "lbs")
print("Escalation :", features.escalation_indicators)

## 7. Run extraction on all notes

Loop through the three notes, extract features, collect the results.


In [ ]:
results = {}
for note_id, text in NOTES.items():
    print(f"Extracting {note_id}...")
    results[note_id] = extract_features(text)

print(f"\nDone — extracted features for {len(results)} notes.")

## 8. View results as a DataFrame

In [ ]:
import pandas as pd

rows = []
for note_id, f in results.items():
    rows.append({
        "note_id": note_id,
        "dx_cluster": f.primary_dx_cluster.value,
        "status": f.overall_status.value,
        "dyspnea": f.dyspnea.value,
        "BP": f"{f.vitals.bp_systolic}/{f.vitals.bp_diastolic}" if f.vitals.bp_systolic else None,
        "spo2": f.vitals.spo2_percent,
        "wt_delta": f.vitals.weight_change_lbs,
        "new_confusion": f.new_confusion,
        "fall": f.fall_since_last_visit,
        "prn_diuretic": f.prn_diuretic_used,
        "adherence_concern": f.adherence_concern,
        "n_escalations": len(f.escalation_indicators),
    })

df = pd.DataFrame(rows).set_index("note_id")
df

## 9. Inspect a single note's full extraction

In [ ]:
import json

note_id = "note_01_hfref"
print(f"=== {note_id} ===\n")
print(json.dumps(results[note_id].model_dump(mode="json"), indent=2))

## 10. Audit trail — every signal traces back to the note

The point of evidence quotes: any extracted feature can be verified against the
source note in one glance. This is what makes the output safe to land in a gold
feature table.


In [ ]:
note_id = "note_01_hfref"
print(f"=== Audit trail: {note_id} ===\n")
for sig in results[note_id].flat_signals:
    print(f"• {sig.name} = {sig.value}")
    print(f"    evidence: \"{sig.evidence_quote}\"\n")

## What's next

To grow this into the full feature-store pipeline:

- **More schema blocks**: wound assessment, OASIS-E items addressed, SDOH details, function/falls.
- **Batch + async**: wrap `extract_features` in `asyncio.gather` for ~4-8x throughput.
- **Retry policy**: on `ValidationError`, retry once on `gemini-2.5-pro`. Log + skip on second failure — never fabricate.
- **MLflow tracing**: log model, latency, tokens, validation errors per call.
- **Pipeline wiring**: stream from `visit_notes_silver` (Spark Structured Streaming) → land into `visit_note_features_gold` with `(note_id, asof_ts)` as PIT key.

See the full `hh-note-features` package for the productionised version.
